In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install thefuzz python-Levenshtein rioxarray shapely
import os
import glob
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray
import warnings
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg') # Forces background plotting, disables inline display
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.interpolate import make_interp_spline
from PIL import Image
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
def process_country_climate(country_name, iso3_code=None):
    print(f"\n\n================ RUNNING CLIMATE PIPELINE: {country_name} ================")
    data_path = f'/content/Processed_Country_Data/{country_name}_DHS_Spatial_Merged.csv' ####### SEE DHS_RAINFALL_CONSOLIDATION FOR HOW TO CREATE THIS CSV
    if not os.path.exists(data_path):
        print(f"No merged dataset found for {country_name}. Skipping.")
        return

    dhs_df = pd.read_csv(data_path)
    dhs_df['interview_date'] = pd.to_datetime(dhs_df['interview_date'])

    precip_dir = f'/content/drive/MyDrive/ERA5_Precip_{country_name}'
    soil_dir = f'/content/drive/MyDrive/ERA5_SoilMoisture_{country_name}'

    has_precip = os.path.exists(precip_dir) and len(glob.glob(f"{precip_dir}/*.tif")) > 0
    has_soil = os.path.exists(soil_dir) and len(glob.glob(f"{soil_dir}/*.tif")) > 0

    if not has_precip and not has_soil:
        print(f"No climate data directories found for {country_name}. Skipping.")
        return

    TARGET_MAP = {
        'informal': 'Sought Informal Help',
        'sought_any': 'Disclosed Violence (Any)',
        'formal': 'Sought Formal Help'
    }

    demographics_cols = ['v012', 'v025', 'v106', 'v190', 'v000', 'v024']
    demographics_formula = "v012 + C(v025) + C(v106) + C(v190) + C(v000) + C(v024)"
    scaler = StandardScaler()

    # Initialize metric storage for saving later
    precip_metrics = []
    soil_metrics = []

    # ---------------------------------------------------------
    # MODULE A: PRECIPITATION
    # ---------------------------------------------------------
    if has_precip:
        print(f"[{country_name}] Processing Precipitation...")
        threshold_bands = [(i, i+5) for i in range(85, 96)] + [(i, 100) for i in range(96, 100)]
        files = sorted(glob.glob(os.path.join(precip_dir, "*.tif")))

        def load_gee_era5_tiff(file_path):
            da = rioxarray.open_rasterio(file_path)
            band_names = da.attrs.get('long_name') or da.long_name
            times = pd.to_datetime([b.split('_')[0] for b in band_names], format='%Y%m%d')
            return da.assign_coords(band=times).rename({'band': 'time', 'x': 'lon', 'y': 'lat'})

        ds_cube = xr.concat([load_gee_era5_tiff(f) for f in files], dim='time')
        baseline_mask = ds_cube.time <= pd.to_datetime('2010-12-31')
        baseline_vals = ds_cube.isel(time=baseline_mask).values
        baseline_vals = baseline_vals[baseline_vals > 0.0001]

        unique_percentiles = sorted(list(set([p for band in threshold_bands for p in band])))
        pct_thresholds = dict(zip(unique_percentiles, np.nanpercentile(baseline_vals, unique_percentiles)))

        clusters = dhs_df[['v001', 'LONGNUM', 'LATNUM']].drop_duplicates()
        lons_da = xr.DataArray(clusters['LONGNUM'].values, dims='cluster')
        lats_da = xr.DataArray(clusters['LATNUM'].values, dims='cluster')

        cluster_cube = ds_cube.interp(lon=lons_da, lat=lats_da, method='nearest')
        cluster_cube['cluster_id'] = ('cluster', clusters['v001'].values)
        cluster_cube = cluster_cube.swap_dims({'cluster': 'cluster_id'})

        # --- OPTIMIZED MODULE A TEMPORAL EXTRACTION ---
        unique_contexts = dhs_df[['v001', 'interview_date']].drop_duplicates()

        for _, row in unique_contexts.iterrows():
            c_id, end_window = row['v001'], row['interview_date']
            start_window = end_window - pd.DateOffset(years=1)
            w_vals = cluster_cube.sel(cluster_id=c_id, time=slice(start_window, end_window)).values

            if len(w_vals) == 0: continue

            metrics = {'v001': c_id, 'interview_date': end_window}
            for lower, upper in threshold_bands:
                metrics[f"exceed_{lower}_{upper}"] = np.sum((w_vals >= pct_thresholds[lower]) & (w_vals <= pct_thresholds[upper]))
            precip_metrics.append(metrics)

        df_precip = dhs_df.merge(pd.DataFrame(precip_metrics), on=['v001', 'interview_date'], how='inner')
        ds_cube.close()

        results_precip = []
        df_precip['weight'] = df_precip['d005'] / 1e6

        for lower, upper in threshold_bands:
            col = f"exceed_{lower}_{upper}"
            if col not in df_precip.columns: continue
            df_precip[f'z_{col}'] = scaler.fit_transform(df_precip[[col]])

            for v_col, v_title in TARGET_MAP.items():
                df_clean = df_precip.dropna(subset=[v_col, f'z_{col}'] + demographics_cols + ['weight'])
                if len(df_clean) < 50: continue

                formula = f"{v_col} ~ z_{col} + {demographics_formula}"
                try:
                    model = smf.glm(formula=formula, data=df_clean, family=sm.families.Binomial(), freq_weights=df_clean['weight'])
                    res = model.fit(cov_type='cluster', cov_kwds={'groups': df_clean['v024']})

                    # --- PROTECTION 1: Skip if standard errors explode ---
                    if np.isnan(res.params[f'z_{col}']) or res.bse[f'z_{col}'] > 10:
                        continue

                    results_precip.append({
                        'Outcome': v_title, 'Lower_Bound': lower,
                        'Coef': res.params[f'z_{col}'],
                        'CI_Lower': res.params[f'z_{col}'] - (1.96 * res.bse[f'z_{col}']),
                        'CI_Upper': res.params[f'z_{col}'] + (1.96 * res.bse[f'z_{col}']),
                        'Significant': res.pvalues[f'z_{col}'] < 0.05
                    })
                except Exception as e:
                    pass

        df_res_p = pd.DataFrame(results_precip)
        if not df_res_p.empty:
            sns.set_theme(style="whitegrid")
            fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
            fig.suptitle(f'{country_name}: Extreme Rainfall & Help-Seeking', fontsize=16, weight='bold')

            for idx, (v_col, v_title) in enumerate(TARGET_MAP.items()):
                ax = axes[idx]
                v_data = df_res_p[df_res_p['Outcome'] == v_title].sort_values('Lower_Bound')
                if v_data.empty: continue

                v_data[['Pct_Change', 'CI_L', 'CI_U']] = (np.exp(v_data[['Coef', 'CI_Lower', 'CI_Upper']]) - 1) * 100
                x, y = v_data['Lower_Bound'], v_data['Pct_Change']

                ax.axhline(0, color='black', linestyle='--')
                ax.fill_between(x, v_data['CI_L'], v_data['CI_U'], color='#1f77b4', alpha=0.2)

                # --- PROTECTION 2: Safe Spline Interpolation ---
                if len(v_data) >= 4:
                    x_smooth = np.linspace(x.min(), x.max(), 300)
                    y_smooth = make_interp_spline(x, y, k=3)(x_smooth)
                    ax.plot(x_smooth, y_smooth, color='#1f77b4')
                else:
                    ax.plot(x, y, color='#1f77b4', linestyle='--')

                ax.scatter(x, y, color=['#d62728' if sig else '#1f77b4' for sig in v_data['Significant']], s=60, edgecolor='white')
                ax.set_title(v_title, weight='bold')

            plt.tight_layout()
            plt.savefig(f'/content/{country_name}_precip.png', dpi=300)
            plt.close()


    # ---------------------------------------------------------
    # MODULE B: SOIL MOISTURE (4 SCENARIOS)
    # ---------------------------------------------------------
    if has_soil:
        print(f"[{country_name}] Processing Soil Moisture...")
        dry_bands = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 10), (0, 15)]
        wet_bands = [(85, 100), (90, 100), (95, 100), (96, 100), (97, 100), (98, 100), (99, 100)]
        threshold_bands = dry_bands + wet_bands
        unique_percentiles = sorted(list(set([p for band in threshold_bands for p in band])))

        def load_gee_soil_tiff(file_path):
            da = rioxarray.open_rasterio(file_path)
            band_names = da.attrs.get('long_name') or da.long_name
            times = pd.to_datetime([b.split('_')[0] for b in band_names])
            layers = ['layer_1' if 'layer_1' in b else 'layer_2' for b in band_names]
            idx_l1 = [i for i, l in enumerate(layers) if l == 'layer_1']
            da_l1 = da.isel(band=idx_l1).assign_coords(band=times[idx_l1]).rename({'band': 'time', 'x': 'lon', 'y': 'lat'})
            return xr.Dataset({'layer_1': da_l1})

        soil_files = sorted(glob.glob(os.path.join(soil_dir, "*.tif")))
        ds_soil = xr.concat([load_gee_soil_tiff(f) for f in soil_files], dim='time').sortby('time')

        clusters = dhs_df[['v001', 'LONGNUM', 'LATNUM']].drop_duplicates()
        lons_da = xr.DataArray(clusters['LONGNUM'].values, dims='cluster')
        lats_da = xr.DataArray(clusters['LATNUM'].values, dims='cluster')

        cluster_soil = ds_soil.interp(lon=lons_da, lat=lats_da, method='linear')
        cluster_soil['cluster_id'] = ('cluster', clusters['v001'].values)
        cluster_soil = cluster_soil.swap_dims({'cluster': 'cluster_id'})

        baseline_mask = cluster_soil.time <= pd.to_datetime('2016-06-30')
        baseline_cube = cluster_soil.isel(time=baseline_mask)

        doy_mean = baseline_cube['layer_1'].groupby('time.dayofyear').mean('time')
        doy_padded = xr.concat([doy_mean.isel(dayofyear=slice(-15, None)), doy_mean, doy_mean.isel(dayofyear=slice(0, 15))], dim='dayofyear')
        doy_smoothed = doy_padded.rolling(dayofyear=30, center=True).mean().isel(dayofyear=slice(15, -15))
        is_historically_wet_doy = doy_smoothed >= doy_smoothed.mean(dim='dayofyear')

        # --- OPTIMIZED MODULE B TEMPORAL EXTRACTION ---
        unique_contexts = dhs_df[['v001', 'interview_date']].drop_duplicates()

        for _, row in unique_contexts.iterrows():
            c_id = row['v001']
            end_window = row['interview_date']
            start_window = end_window - pd.DateOffset(years=1)

            c_data = cluster_soil.sel(cluster_id=c_id)
            c_baseline = baseline_cube.sel(cluster_id=c_id)
            c_wet_mask = is_historically_wet_doy.sel(cluster_id=c_id)

            b_vals = c_baseline['layer_1'].values
            b_doys = c_baseline.time.dt.dayofyear.values
            b_is_wet = c_wet_mask.values[b_doys - 1]

            wet_b_vals = b_vals[b_is_wet]
            dry_b_vals = b_vals[~b_is_wet]

            pct_wet = np.nanpercentile(wet_b_vals, unique_percentiles) if len(wet_b_vals) > 0 else np.nanpercentile(b_vals, unique_percentiles)
            pct_dry = np.nanpercentile(dry_b_vals, unique_percentiles) if len(dry_b_vals) > 0 else np.nanpercentile(b_vals, unique_percentiles)

            wet_thresh = dict(zip(unique_percentiles, pct_wet))
            dry_thresh = dict(zip(unique_percentiles, pct_dry))

            window_data = c_data.sel(time=slice(start_window, end_window))
            w_vals = window_data['layer_1'].values
            w_doys = window_data.time.dt.dayofyear.values

            if len(w_vals) == 0: continue
            w_is_wet = c_wet_mask.values[w_doys - 1]
            w_is_dry = ~w_is_wet

            metrics = {'v001': c_id, 'interview_date': end_window}
            for lower, upper in wet_bands:
                metrics[f"Wet_in_Wet_{lower}_{upper}"] = np.sum((w_vals >= wet_thresh[lower]) & (w_vals <= wet_thresh[upper]) & w_is_wet)
                metrics[f"Wet_in_Dry_{lower}_{upper}"] = np.sum((w_vals >= dry_thresh[lower]) & (w_vals <= dry_thresh[upper]) & w_is_dry)
            for lower, upper in dry_bands:
                metrics[f"Dry_in_Dry_{lower}_{upper}"] = np.sum((w_vals >= dry_thresh[lower]) & (w_vals <= dry_thresh[upper]) & w_is_dry)
                metrics[f"Dry_in_Wet_{lower}_{upper}"] = np.sum((w_vals >= wet_thresh[lower]) & (w_vals <= wet_thresh[upper]) & w_is_wet)

            soil_metrics.append(metrics)

        ds_soil.close()
        df_soil = dhs_df.merge(pd.DataFrame(soil_metrics), on=['v001', 'interview_date'], how='inner')
        df_soil['weight'] = df_soil['d005'] / 1e6

        results_soil = []
        analysis_bands = (
            [(b[0], b[1], "Wet_in_Wet", "Extreme Wet in Wet Season") for b in wet_bands] +
            [(b[0], b[1], "Wet_in_Dry", "Extreme Wet in Dry Season") for b in wet_bands] +
            [(b[0], b[1], "Dry_in_Dry", "Extreme Dry in Dry Season") for b in dry_bands] +
            [(b[0], b[1], "Dry_in_Wet", "Extreme Dry in Wet Season") for b in dry_bands]
        )

        for lower, upper, prefix, season_title in analysis_bands:
            col = f"{prefix}_{lower}_{upper}"
            if col not in df_soil.columns: continue
            df_soil[f'z_{col}'] = scaler.fit_transform(df_soil[[col]])

            for v_col, v_title in TARGET_MAP.items():
                df_clean = df_soil.dropna(subset=[v_col, f'z_{col}'] + demographics_cols + ['weight'])
                if len(df_clean) < 50: continue

                formula = f"{v_col} ~ z_{col} + {demographics_formula}"
                try:
                    model = smf.glm(formula=formula, data=df_clean, family=sm.families.Binomial(), freq_weights=df_clean['weight'])
                    res = model.fit(cov_type='cluster', cov_kwds={'groups': df_clean['v024']})

                    # --- PROTECTION 1: Skip if standard errors explode ---
                    if np.isnan(res.params[f'z_{col}']) or res.bse[f'z_{col}'] > 10:
                        continue

                    results_soil.append({
                        'Outcome': v_title,
                        'Scenario': season_title,
                        'Bound_Plot_Value': lower if 'Wet_' in prefix else upper,
                        'Coef': res.params[f'z_{col}'],
                        'CI_Lower': res.params[f'z_{col}'] - (1.96 * res.bse[f'z_{col}']),
                        'CI_Upper': res.params[f'z_{col}'] + (1.96 * res.bse[f'z_{col}']),
                        'Significant': res.pvalues[f'z_{col}'] < 0.05
                    })
                except Exception as e:
                    pass

        df_res_s = pd.DataFrame(results_soil)
        if not df_res_s.empty:
            sns.set_theme(style="whitegrid")
            fig, axes = plt.subplots(4, 3, figsize=(18, 20), sharey=False)
            fig.suptitle(f'{country_name}: Extreme Soil Moisture & Help-Seeking (4 Scenarios)', fontsize=18, weight='bold', y=1.02)

            scenarios = [
                'Extreme Wet in Wet Season', 'Extreme Wet in Dry Season',
                'Extreme Dry in Wet Season', 'Extreme Dry in Dry Season'
            ]
            colors = {
                'Extreme Wet in Wet Season': ('#1f77b4', '#d62728'),
                'Extreme Wet in Dry Season': ('#2ca02c', '#d62728'),
                'Extreme Dry in Wet Season': ('#8c564b', '#d62728'),
                'Extreme Dry in Dry Season': ('#e6550d', '#8b0000')
            }

            for row_idx, scenario in enumerate(scenarios):
                line_c, sig_c = colors[scenario]
                for col_idx, (v_col, v_title) in enumerate(TARGET_MAP.items()):
                    ax = axes[row_idx, col_idx]
                    v_data = df_res_s[(df_res_s['Outcome'] == v_title) & (df_res_s['Scenario'] == scenario)].sort_values('Bound_Plot_Value')

                    if v_data.empty:
                        ax.set_title(f"Insufficient Data", fontsize=10)
                        continue

                    v_data[['Pct_Change', 'CI_L', 'CI_U']] = (np.exp(v_data[['Coef', 'CI_Lower', 'CI_Upper']]) - 1) * 100
                    x, y = v_data['Bound_Plot_Value'], v_data['Pct_Change']

                    ax.axhline(0, color='black', linestyle='--')
                    ax.fill_between(x, v_data['CI_L'], v_data['CI_U'], color=line_c, alpha=0.15)

                    # --- PROTECTION 2: Safe Spline Interpolation ---
                    if len(v_data) >= 4:
                        x_smooth = np.linspace(x.min(), x.max(), 300)
                        y_smooth = make_interp_spline(x, y, k=3)(x_smooth)
                        ax.plot(x_smooth, y_smooth, color=line_c, linewidth=2)
                    else:
                        ax.plot(x, y, color=line_c, linewidth=2, linestyle='--')

                    ax.scatter(x, y, color=[sig_c if sig else line_c for sig in v_data['Significant']], s=60, edgecolor='white')

                    ax.set_title(f"{v_title}\n{scenario}", fontsize=12, weight='bold')
                    ax.set_xlabel('Lower Bound (%)' if 'Wet' in scenario.split()[1] else 'Upper Bound (%)', fontsize=10)

            plt.tight_layout()
            plt.savefig(f'/content/{country_name}_soil.png', dpi=300, bbox_inches='tight')
            plt.close()


    # ---------------------------------------------------------
    # MODULE C: STITCHING PRECIPITATION & SOIL MOISTURE
    # ---------------------------------------------------------
    if has_precip and has_soil:
        try:
            precip_img_path = f'/content/{country_name}_precip.png'
            soil_img_path = f'/content/{country_name}_soil.png'

            if os.path.exists(precip_img_path) and os.path.exists(soil_img_path):
                img_p = Image.open(precip_img_path)
                img_s = Image.open(soil_img_path)

                target_width = img_s.width
                precip_ratio = target_width / img_p.width
                new_precip_height = int(img_p.height * precip_ratio)
                img_p_resized = img_p.resize((target_width, new_precip_height), Image.Resampling.LANCZOS)

                combined = Image.new('RGB', (target_width, new_precip_height + img_s.height), (255, 255, 255))
                combined.paste(img_p_resized, (0, 0))
                combined.paste(img_s, (0, new_precip_height))

                out_stitched_path = f'/content/{country_name}_combined_sensitivity.png'
                combined.save(out_stitched_path)
                print(f"[{country_name}] Successfully generated stitched master visualization: {out_stitched_path}")
        except Exception as e:
            print(f"[{country_name}] Failed to stitch plots: {e}")

    # ---------------------------------------------------------
    # MODULE D: SAVE UPDATED DATASET (NEW ADDITION)
    # ---------------------------------------------------------
    final_df = dhs_df.copy()

    if has_precip and len(precip_metrics) > 0:
        final_df = final_df.merge(
            pd.DataFrame(precip_metrics).drop_duplicates(subset=['v001', 'interview_date']),
            on=['v001', 'interview_date'], how='left'
        )

    if has_soil and len(soil_metrics) > 0:
        final_df = final_df.merge(
            pd.DataFrame(soil_metrics).drop_duplicates(subset=['v001', 'interview_date']),
            on=['v001', 'interview_date'], how='left'
        )

    final_df.to_csv(data_path, index=False)
    print(f"[{country_name}] Successfully saved updated dataset with climate metrics to {data_path}")

# Run for all available countries

country_mapping = {
    'Benin': 'Benin', 'Burkina Faso': 'Burkina Faso', 'Cote dIvoire': "Cote d'Ivoire",
    'Gambia': 'Gambia', 'Ghana': 'Ghana', 'Liberia': 'Liberia', 'Mali': 'Mali',
    'Mauritania': 'Mauritania', 'Nigeria': 'Nigeria', 'Senegal': 'Senegal',
    'Sierra Leone': 'Sierra Leone', 'Togo': 'Togo', 'Angola': 'Angola',
    'Cameroon': 'Cameroon', 'Chad': 'Chad', 'DR Congo': 'DR Congo',
    'Gabon': 'Gabon', 'Burundi': 'Burundi', 'Comoros': 'Comoros', 'Ethiopia': 'Ethiopia',
    'Kenya': 'Kenya', 'Madagascar': 'Madagascar', 'Malawi': 'Malawi',
    'Mozambique': 'Mozambique', 'Rwanda': 'Rwanda', 'Tanzania': 'Tanzania',
    'Uganda': 'Uganda', 'Zambia': 'Zambia', 'Zimbabwe': 'Zimbabwe', 'Lesotho': 'Lesotho',
    'Namibia': 'Namibia', 'South Africa': 'South Africa'
}

# (Assuming `country_mapping` is defined earlier in your notebook)
for country_name in country_mapping.keys():
    try:
        process_country_climate(country_name)
    except Exception as e:
        print(f"Failed to complete pipeline for {country_name}: {e}")